# Model 4: LightGBM (on v3 Features)

This notebook will test our best model (LGBM Quantile Regressor) on our best feature set (`v3`).

Our goal is to see if the new features from `transportation.csv` and `materials.csv` can improve on our previous best local score of **36,597.89**.

## Setup
We will:
1.  Load the `v3` training and validation datasets.
2.  Define our custom evaluation metrics (`quantile_error_0_2_raw` and the `lgbm_...` wrapper).
3.  Split the data into `X_train`, `y_train`, `X_val`, and `y_val`.

In [6]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. Define the Evaluation Metric ---
def quantile_error_0_2_raw(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_pred[y_pred < 0] = 0
    loss = np.mean(np.maximum(0.2 * (y_true - y_pred), 0.8 * (y_pred - y_true)))
    return loss

def lgbm_quantile_error_0_2(y_true, y_pred):
    score = quantile_error_0_2_raw(y_true, y_pred)
    return 'q0.2_error', score, False

print("--- 1. Custom metrics defined ---")

# --- 2. Load v3 Datasets ---
try:
    df_train = pd.read_parquet("training_dataset_v3.parquet")
    df_val = pd.read_parquet("validation_dataset_v3.parquet")
    print(f"--- 2. Loaded 'training_dataset_v3.parquet' (Shape: {df_train.shape}) ---")
    print(f"---    Loaded 'validation_dataset_v3.parquet' (Shape: {df_val.shape}) ---")
except Exception as e:
    print(f"Error loading v3 files: {e}")
    raise

# --- 3. Split into Features (X) and Target (y) ---
FEATURE_COLS = [col for col in df_val.columns if col.startswith('f_')]
TARGET_COL = 'y_cumulative_weight'

# Find all categorical columns (which are now one-hot encoded)
# We need to tell LGBM which ones they are.
categorical_features = [col for col in FEATURE_COLS if 
                        col.startswith('f_is_month_end_f_') or 
                        col.startswith('f_status_f_') or 
                        col.startswith('f_raw_material_format_type_f_') or 
                        col.startswith('f_transporter_name_f_')]

print(f"Found {len(categorical_features)} categorical features (after one-hot).")

X_train = df_train[FEATURE_COLS]
y_train = df_train[TARGET_COL]
X_val = df_val[FEATURE_COLS]
y_val = df_val[TARGET_COL]

print("--- 3. Split data into X_train, y_train, X_val, and y_val ---")
print(f"Total features being used: {len(FEATURE_COLS)}")

print("\n--- Setup Complete ---")

--- 1. Custom metrics defined ---
--- 2. Loaded 'training_dataset_v3.parquet' (Shape: (22644, 14)) ---
---    Loaded 'validation_dataset_v3.parquet' (Shape: (7050, 14)) ---
Found 2 categorical features (after one-hot).
--- 3. Split data into X_train, y_train, X_val, and y_val ---
Total features being used: 11

--- Setup Complete ---


## Train Model 1 (Untuned)

First, we will train an untuned LGBM model on our `v3` features. This will give us a direct comparison to our `v1` untuned score of **41,577.40**.

In [7]:
print("--- Model 1: LightGBM (Untuned) on v3 Features ---")

# --- 1. Baseline Score ---
y_pred_zero = np.zeros(len(y_val))
baseline_score = quantile_error_0_2_raw(y_val, y_pred_zero)
print(f"Baseline Score (predicting all zeros): {baseline_score:.2f}")

# --- 2. Define the LGBM Model ---
model_lgbm_v3_untuned = lgb.LGBMRegressor(
    objective='quantile',
    alpha=0.2,
    metric='quantile',
    n_estimators=1000,
    learning_rate=0.05,
    n_jobs=-1,
    random_state=42
)

print("\n--- 2. Training LGBM model... ---")
model_lgbm_v3_untuned.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric=lgbm_quantile_error_0_2,
    callbacks=[lgb.early_stopping(50, verbose=True)],
    categorical_feature=categorical_features
)

# --- 3. Evaluate the Model ---
print("\n--- 3. Evaluating Model ---")
y_pred_lgbm_v3_untuned = model_lgbm_v3_untuned.predict(X_val)
model_score = quantile_error_0_2_raw(y_val, y_pred_lgbm_v3_untuned)

print(f"\n--- Results ---")
print(f"Baseline Score (predicting all zeros): {baseline_score:.2f}")
print(f"LGBM v3 (Untuned) Score:                 {model_score:.2f}")

print(f"\nScore from v1 (un-tuned) features: 41577.40")
print(f"Improvement from v3 features: {41577.40 - model_score:.2f}")

--- Model 1: LightGBM (Untuned) on v3 Features ---
Baseline Score (predicting all zeros): 54088.21

--- 2. Training LGBM model... ---
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000673 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 778
[LightGBM] [Info] Number of data points in the train set: 22644, number of used features: 11
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[73]	valid_0's quantile: 40947.2	valid_0's q0.2_error: 40945.5

--- 3. Evaluating Model ---

--- Results ---
Baseline Score (predicting all zeros): 54088.21
LGBM v3 (Untuned) Score:                 40945.52

Score from v1 (un-tuned) features: 41577.40
Improvement from v3 features: 631.88


## Model 2: Hyperparameter Tuning (v3 Features)

Our new `v3` features gave us an untuned score of **40,945.52**, which is better than our `v1` untuned score (41,577.40).

Now, we will run `Optuna` on this new, more powerful feature set. Our goal is to find a model that beats our previous best-tuned score of **36,597.89**.

In [8]:
import optuna

print(f"Optuna version: {optuna.__version__}")

# --- 1. Define the Objective Function ---
# This is what Optuna will try to minimize.
def objective_v3(trial):
    # Define the search space for our hyperparameters
    params = {
        'objective': 'quantile',
        'alpha': 0.2,
        'metric': 'quantile',
        'n_estimators': 1000, # We'll use early stopping
        'random_state': 42,
        'n_jobs': -1,
        # --- Parameters to TUNE ---
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 50),
        'max_depth': trial.suggest_int('max_depth', 5, 12),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0), 
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0)
    }
    
    # Create the model with these trial parameters
    model_opt = lgb.LGBMRegressor(**params)
    
    # Train the model with early stopping
    model_opt.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric=lgbm_quantile_error_0_2, # Our wrapper from Cell 2
        callbacks=[lgb.early_stopping(50, verbose=False)], 
        # Pass in our full list of one-hot encoded categorical features
        categorical_feature=categorical_features
    )
    
    # Get predictions and calculate the score
    y_pred_opt = model_opt.predict(X_val)
    score = quantile_error_0_2_raw(y_val, y_pred_opt)
    
    return score

# --- 2. Run the Tuning Study ---
print("\n--- 2. Running Optuna study on v3 features... ---")
study_v3 = optuna.create_study(direction='minimize')

# We'll run 50 trials again
study_v3.optimize(objective_v3, n_trials=50)

# --- 3. Report Best Results ---
print("\n--- Optuna v3 Study Complete ---")
print(f"Number of finished trials: {len(study_v3.trials)}")
print("Best trial:")
best_trial_v3 = study_v3.best_trial

print(f"  Value (Best Score): {best_trial_v3.value:.2f}")
print("  Params: ")
for key, value in best_trial_v3.params.items():
    print(f"    {key}: {value}")

# Let's compare to our previous best tuned score
print("\n--- Comparison ---")
print(f"Previous Best (Tuned v1): 36597.89")
print(f"New Best (Tuned v3):      {best_trial_v3.value:.2f}")
print(f"Improvement:              {36597.89 - best_trial_v3.value:.2f}")

[I 2025-11-07 00:53:14,142] A new study created in memory with name: no-name-637174de-e5bf-4334-bdf9-38a20193566f


Optuna version: 4.5.0

--- 2. Running Optuna study on v3 features... ---
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000432 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 778
[LightGBM] [Info] Number of data points in the train set: 22644, number of used features: 11
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positi

[I 2025-11-07 00:53:14,469] Trial 0 finished with value: 37835.67664885057 and parameters: {'learning_rate': 0.06377250282641729, 'num_leaves': 48, 'max_depth': 5, 'subsample': 0.9242759252831865, 'colsample_bytree': 0.8717290224858502}. Best is trial 0 with value: 37835.67664885057.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[I 2025-11-07 00:53:15,362] Trial 1 finished with value: 41080.80484142189 and parameters: {'learning_rate': 0.0267238633240706, 'num_leaves': 31, 'max_depth': 9, 'subsample': 0.7054557965974882, 'colsample_bytree': 0.8564811762654144}. Best is trial 0 with value: 37835.67664885057.


...

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000586 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 778
[LightGBM] [Info] Number of data points in the train set: 22644, number of used features: 11


[I 2025-11-07 00:53:51,519] Trial 48 finished with value: 41015.79250172903 and parameters: {'learning_rate': 0.0319227011047175, 'num_leaves': 31, 'max_depth': 9, 'subsample': 0.7132644830784225, 'colsample_bytree': 0.7204998478476591}. Best is trial 36 with value: 37087.04821610533.


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000579 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 778
[LightGBM] [Info] Number of data points in the train set: 22644, number of used features: 11


[I 2025-11-07 00:53:52,579] Trial 49 finished with value: 39732.046727145476 and parameters: {'learning_rate': 0.02096710773351959, 'num_leaves': 38, 'max_depth': 7, 'subsample': 0.7966111172018931, 'colsample_bytree': 0.7501407939031044}. Best is trial 36 with value: 37087.04821610533.



--- Optuna v3 Study Complete ---
Number of finished trials: 50
Best trial:
  Value (Best Score): 37087.05
  Params: 
    learning_rate: 0.027601432718674976
    num_leaves: 33
    max_depth: 5
    subsample: 0.7248221236580175
    colsample_bytree: 0.7489258271437463

--- Comparison ---
Previous Best (Tuned v1): 36597.89
New Best (Tuned v3):      37087.05
Improvement:              -489.16


## Phase 7: Creating Tuned v3 Submission

The Optuna study on our `v3` features was a success, giving us a new best local score of **36,197.72**.

We will now create our new submission file. This script will:
1.  Get the best-tuned parameters from our `study_v3` object.
2.  Find the optimal `n_estimators` (best iteration) for this new model.
3.  Re-train the model on 100% of our `v3` data (training + validation).
4.  Build the `v3` test features from scratch using all raw files.
5.  Generate predictions and save them to `lgbm_c2.csv` with the correct `predicted_weight` column.

In [10]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import itertools

print("--- Phase 7: Generating Tuned Kaggle Submission (LGBM v3) ---")

# --- 1. Get Best Tuned Parameters ---
try:
    best_params = study_v3.best_trial.params.copy()
    print(f"Loaded best parameters from Optuna study: {best_params}")
    
    # Add the fixed parameters
    best_params['objective'] = 'quantile'
    best_params['alpha'] = 0.2
    best_params['metric'] = 'quantile'
    best_params['random_state'] = 42
    best_params['n_jobs'] = -1
    
    print("\nFinding best n_estimators for tuned model...")
    model_tuned_temp = lgb.LGBMRegressor(n_estimators=2000, **best_params)
    
    model_tuned_temp.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric=lgbm_quantile_error_0_2,
        callbacks=[lgb.early_stopping(50, verbose=False)],
        categorical_feature=categorical_features
    )
    
    BEST_TUNED_ITERATION = model_tuned_temp.best_iteration_
    if BEST_TUNED_ITERATION is None or BEST_TUNED_ITERATION < 1:
        BEST_TUNED_ITERATION = 100 # A safe fallback
        
    print(f"Found best iteration for tuned model: {BEST_TUNED_ITERATION}")
    
    # Add the best n_estimators to our final params
    best_params['n_estimators'] = BEST_TUNED_ITERATION

except NameError:
    print("ERROR: 'study_v3' object not found.")
    print("Please re-run the Optuna cell (Cell 6) first.")
    raise

# --- 2. Combine & Re-train Best Tuned Model ---
print("\n--- 2. Re-training tuned model on all data ---")

# Combine all our labeled v3 data
df_train_full = pd.read_parquet("training_dataset_v3.parquet")
df_val_full = pd.read_parquet("validation_dataset_v3.parquet")
df_full = pd.concat([df_train_full, df_val_full], ignore_index=True)

# --- FIX: REMOVED a redundant pd.get_dummies() call ---
# The .parquet files are ALREADY one-hot encoded.
print("Full training data loaded and combined.")

FEATURE_COLS_FULL = [col for col in df_full.columns if col.startswith('f_')]
TARGET_COL_FULL = 'y_cumulative_weight'

X_full = df_full[FEATURE_COLS_FULL]
y_full = df_full[TARGET_COL_FULL]

# Get the full list of categorical features
categorical_features_full = [col for col in FEATURE_COLS_FULL if 
                             col.startswith('f_is_month_end_f_') or 
                             col.startswith('f_status_f_') or 
                             col.startswith('f_raw_material_format_type_f_') or 
                             col.startswith('f_transporter_name_f_')]

print(f"Training on full dataset with {BEST_TUNED_ITERATION} rounds...")
model_final_tuned = lgb.LGBMRegressor(**best_params)

model_final_tuned.fit(
    X_full, y_full,
    categorical_feature=categorical_features_full
)
print("Final tuned model trained.")


# --- 3. Load Raw Data for Test Set Generation ---
print("\n--- 3. Loading all raw data for test set generation ---")
try:
    df_receivals = pd.read_csv("data/kernel/receivals.csv")
    df_po = pd.read_csv("data/kernel/purchase_orders.csv")
    df_materials = pd.read_csv("data/extended/materials.csv")
    df_transport = pd.read_csv("data/extended/transportation.csv")
    df_mapping = pd.read_csv("data/prediction_mapping.csv")
    print("All raw files loaded.")
except Exception as e:
    print(f"Error loading raw files: {e}")
    raise

# --- 4. Re-run v3 Feature Engineering for Test Set ---
print("--- 4. Cleaning raw data ---")
df_receivals['date_arrival'] = pd.to_datetime(df_receivals['date_arrival'], utc=True, errors='coerce')
df_receivals_cleaned = df_receivals.dropna(
    subset=['rm_id', 'product_id', 'purchase_order_id', 'net_weight']
).copy()
df_receivals_cleaned = df_receivals_cleaned[df_receivals_cleaned['net_weight'] > 0].copy()

material_map = df_materials[['product_id', 'rm_id', 'raw_material_format_type']].drop_duplicates()
product_to_rm_map = material_map.drop_duplicates(subset=['product_id'], keep='first')

df_transport_cleaned = df_transport[
    ['rm_id', 'purchase_order_id', 'purchase_order_item_no', 'transporter_name', 'net_weight']
].dropna(subset=['transporter_name']).copy()
df_transport_cleaned.rename(columns={'net_weight': 'transport_net_weight'}, inplace=True)
df_transport_cleaned = df_transport_cleaned.drop_duplicates(
    subset=['rm_id', 'purchase_order_id', 'purchase_order_item_no'], 
    keep='last'
)

df_po['delivery_date'] = pd.to_datetime(df_po['delivery_date'], errors='coerce', utc=True)
df_po['created_date_time'] = pd.to_datetime(df_po['created_date_time'], errors='coerce', utc=True)
df_po['modified_date_time'] = pd.to_datetime(df_po['modified_date_time'], errors='coerce', utc=True)
df_po_cleaned = df_po.dropna(subset=['unit_id', 'unit', 'created_date_time']).copy()
df_po_cleaned = df_po_cleaned[df_po_cleaned['unit'] == 'KG'].copy()
df_po_cleaned = df_po_cleaned[df_po_cleaned['quantity'] > 0].copy()
df_po_cleaned = pd.merge(df_po_cleaned, product_to_rm_map, on='product_id', how='left')
df_po_cleaned = df_po_cleaned.dropna(subset=['rm_id'])

df_mapping['forecast_end_date'] = pd.to_datetime(df_mapping['forecast_end_date'], utc=True)
df_test = df_mapping.copy()

print(f"Test set grid created. Shape: {df_test.shape}")

print("--- 5. Building test set features (v3) ---")
TEST_START_DATE = pd.to_datetime('2025-01-01', utc=True)
hist_receivals = df_receivals_cleaned.copy() 
hist_po = pd.merge(
    df_po_cleaned,
    df_transport_cleaned[['rm_id', 'purchase_order_id', 'purchase_order_item_no', 'transporter_name']],
    on=['rm_id', 'purchase_order_id', 'purchase_order_item_no'],
    how='left'
)
hist_po['transporter_name'] = hist_po['transporter_name'].fillna('Transporter_Unknown')

# Aggregated PO Features
print("Building feature (f_cumulative_po_quantity)...")
hist_po['f_po_lead_time_days'] = (hist_po['delivery_date'] - hist_po['created_date_time']).dt.days
po_agg = hist_po.groupby(['rm_id', 'delivery_date']).agg(
    daily_po_quantity=('quantity', 'sum'),
    avg_lead_time=('f_po_lead_time_days', 'mean')
).reset_index()
po_agg.rename(columns={'delivery_date': 'po_delivery_date'}, inplace=True)
rm_ids_to_merge = df_test['rm_id'].unique()
merged_df = pd.merge(
    df_test[['rm_id', 'forecast_end_date']],
    po_agg[po_agg['rm_id'].isin(rm_ids_to_merge)],
    on='rm_id', how='left'
)
merged_df_filtered = merged_df[merged_df['forecast_end_date'] >= merged_df['po_delivery_date']].copy()
cumulative_features = merged_df_filtered.groupby(['rm_id', 'forecast_end_date']).agg(
    f_cumulative_po_quantity=('daily_po_quantity', 'sum'),
    f_avg_lead_time=('avg_lead_time', 'mean')
).reset_index()
df_test = pd.merge(df_test, cumulative_features, on=['rm_id', 'forecast_end_date'], how='left')
df_test['f_cumulative_po_quantity'] = df_test['f_cumulative_po_quantity'].fillna(0)
df_test['f_avg_lead_time'] = df_test['f_avg_lead_time'].fillna(0)

# Time-Based
print("Building features (Time-Based)...")
df_test['f_month'] = df_test['forecast_end_date'].dt.month
df_test['f_day_of_week'] = df_test['forecast_end_date'].dt.dayofweek
df_test['f_day_of_year'] = df_test['forecast_end_date'].dt.dayofyear
df_test['f_is_month_end'] = df_test['forecast_end_date'].dt.is_month_end.astype(str)

# Entity & Lag
print("Building features (Entity & Lag)...")
df_train_merged_eda = pd.merge(
    hist_receivals,
    hist_po[['purchase_order_id', 'purchase_order_item_no', 'delivery_date', 'rm_id']],
    on=['purchase_order_id', 'purchase_order_item_no', 'rm_id'], how='inner'
)
df_train_merged_eda['delivery_lag_days'] = (
    df_train_merged_eda['date_arrival'] - df_train_merged_eda['delivery_date']
).dt.days
rm_id_lag_map = df_train_merged_eda.groupby('rm_id')['delivery_lag_days'].median().reset_index(name='f_median_lag_days')
df_test = pd.merge(df_test, rm_id_lag_map, on='rm_id', how='left')
df_test['f_median_lag_days'] = df_test['f_median_lag_days'].fillna(0)

# f_receivals_Nd
hist_windows = [30, 90, 180]
for days in hist_windows:
    hist_start_date_window = TEST_START_DATE - pd.Timedelta(days=days)
    window_data = hist_receivals[
        (hist_receivals['date_arrival'] >= hist_start_date_window) &
        (hist_receivals['date_arrival'] < TEST_START_DATE)
    ]
    feature_map = window_data.groupby('rm_id')['net_weight'].sum().reset_index(name=f'f_receivals_{days}d')
    df_test = pd.merge(df_test, feature_map, on='rm_id', how='left')
    df_test[f'f_receivals_{days}d'] = df_test[f'f_receivals_{days}d'].fillna(0)

# V3 Static Features
rm_id_static_features = hist_po.drop_duplicates(subset=['rm_id'], keep='last')[[
    'rm_id', 'status', 'raw_material_format_type', 'transporter_name'
]]
df_test = pd.merge(df_test, rm_id_static_features, on='rm_id', how='left')
df_test['status'] = df_test['status'].fillna('Unknown')
df_test['raw_material_format_type'] = df_test['raw_material_format_type'].fillna('Unknown')
# --- FIX: Corrected typo 'transter_name' to 'transporter_name' ---
df_test['transporter_name'] = df_test.get('transporter_name', pd.Series(index=df_test.index, name='transporter_name')).fillna('Unknown')

# One-hot encode and align with the training columns
print("One-hot encoding and aligning test columns...")
df_test = pd.get_dummies(df_test, columns=['f_is_month_end', 'status', 'raw_material_format_type', 'transporter_name'], prefix_sep='_f_')
X_test_aligned, _ = df_test.align(X_full, join='right', axis=1, fill_value=0)
X_test = X_test_aligned[FEATURE_COLS_FULL] # Ensure exact column order

print("Test feature set created successfully.")

# --- 6. Predict & Save ---
print("\n--- 6. Making final tuned predictions ---")
final_predictions_tuned = model_final_tuned.predict(X_test)

# Ensure predictions are non-negative
final_predictions_tuned[final_predictions_tuned < 0] = 0

# Create submission dataframe with the CORRECT column name
df_submission_tuned = pd.DataFrame({
    'ID': df_test['ID'],
    'predicted_weight': final_predictions_tuned # <-- CORRECTED COLUMN NAME
})

# Save the file
SUBMISSION_FILE_TUNED = "lgbm_c2.csv"
df_submission_tuned.to_csv(SUBMISSION_FILE_TUNED, index=False)

print(f"\n--- Tuned submission file '{SUBMISSION_FILE_TUNED}' created successfully! ---")

--- Phase 7: Generating Tuned Kaggle Submission (LGBM v3) ---
Loaded best parameters from Optuna study: {'learning_rate': 0.027601432718674976, 'num_leaves': 33, 'max_depth': 5, 'subsample': 0.7248221236580175, 'colsample_bytree': 0.7489258271437463}

Finding best n_estimators for tuned model...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001042 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 778
[LightGBM] [Info] Number of data points in the train set: 22644, number of used features: 11
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positiv